# 09b LightGBM after breaks

In 09a, retraining LightGBM only on data after a shift made it slightly worse. Maybe that is because throwing all
the history away is too blunt. Here we try three gentler ways to react, on the same products, origins and shifts
(always the ones known at the origin):

(a) restart: train only from the latest change week (the 09a retrained version);
(b) down-weight: keep all the data, but give rows before the change less weight, halving every 182 days;
(c) shift feature: keep all the data and add two inputs, "a shift has happened" and "weeks since the shift"
    (capped at 104);
(d) blend: the average of the static and restart forecasts.

Products without a shift keep the static forecast. We decided beforehand that a version "helps" only if the
Holm-corrected Wilcoxon p is below 0.05, the median change in MASE is below 0, and the gain is at least 0.0074
(1% of the static error). We test it on all shifts and on the high-confidence ones from 08c.

In [1]:
import sys, time
sys.path.append('../src')
import numpy as np
import pandas as pd

from break_detection import to_weekly, weekly_ca_prices
from calibration import pelt_changepoints
from break_quality import assess_break_at_origin
from forecasting import build_lgb_panel, lgb_forecast
from metrics import paired_test, holm_adjust

BURN_IN, REF_LEN = 8, 52
PRACTICAL = 0.0074
PEN = float(pd.read_csv("../outputs/tables/break_calibration.csv").set_index('method').loc['PELT', 'value'])

fc = pd.read_parquet("../data/processed/backtest_forecasts.parquet")
lgb_fc = fc[fc['model'] == 'lightgbm'][['item_id', 'origin', 'date', 'variant', 'forecast', 'actual']]
err = pd.read_csv("../data/processed/backtest_errors.csv", parse_dates=['origin'])
scales = err[['item_id', 'origin', 'mase_scale']].drop_duplicates()
windows = pd.read_csv("../data/processed/breaks_by_origin.csv",
                      parse_dates=['origin', 'first_date', 'last_change_week', 'last_alarm_week', 'adaptive_start', 'placebo_start'])
ORIGINS = sorted(windows['origin'].unique())
EVAL = ORIGINS[1:]

long_df = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=['date'])
                     for c in ['foods', 'hobbies', 'household']], ignore_index=True).sort_values(['item_id', 'date'])
category = long_df.groupby('item_id')['item_id'].first().str.split('_').str[0]
daily = {i: g[['date', 'sales']].reset_index(drop=True) for i, g in long_df.groupby('item_id')}
print(len(daily), "items;", "eval origins:", [o.date() for o in EVAL])

726 items; eval origins: [datetime.date(2015, 12, 6), datetime.date(2016, 1, 3), datetime.date(2016, 1, 31), datetime.date(2016, 2, 28), datetime.date(2016, 3, 27), datetime.date(2016, 4, 24)]


## The high-confidence shifts at each origin

08c works these out at every origin with data up to that origin only; we read its table.

In [2]:
hc = pd.read_csv("../outputs/tables/hc_by_origin.csv", parse_dates=["origin"])
print(f"item-origins with a break: {len(hc)}; high-confidence: {int(hc['high_confidence'].sum())} "
      f"({hc['high_confidence'].mean():.1%}); items high-confidence at >=1 origin: {hc.loc[hc['high_confidence'], 'item_id'].nunique()}")
print(hc[['agreement', 'effect', 'persistence']].mean().round(3).to_string())

item-origins with a break: 2869; high-confidence: 1172 (40.9%); items high-confidence at >=1 origin: 254
agreement      0.648
effect         0.872
persistence    0.650


## A check first: the refactored LightGBM gives the same forecasts as 09a

In [3]:
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=['date'])
prices = pd.read_csv("../data/raw/sell_prices.csv", dtype={'store_id': 'category'})
price_wk = weekly_ca_prices(prices, calendar, long_df['item_id'].unique())
del prices
panel = build_lgb_panel(long_df, calendar, price_wk)

check, _ = lgb_forecast(panel, EVAL[-1])
saved = lgb_fc[(lgb_fc['origin'] == EVAL[-1]) & (lgb_fc['variant'] == 'static')]
m = check.merge(saved, on=['item_id', 'date'], suffixes=('_new', '_saved'))
print(f"rows compared: {len(m)}, max abs difference: {np.abs(m['forecast_new'] - m['forecast_saved']).max():.2e}")

rows compared: 20328, max abs difference: 0.00e+00


## Training versions (b) and (c) at each scored origin

In [4]:
parts = []
t0 = time.time()
for origin in EVAL:
    w = windows[(windows['origin'] == origin) & windows['has_break']]
    brk = dict(zip(w['item_id'], w['last_change_week']))
    for variant, kw in [('decay', {'decay_break_by_item': brk}), ('level_shift', {'shift_break_by_item': brk})]:
        out, _ = lgb_forecast(panel, origin, **kw)
        out['origin'], out['variant'] = origin, variant
        parts.append(out)
    print(f"{origin.date()} done ({time.time() - t0:.0f}s)")
new = pd.concat(parts, ignore_index=True)

ev = lgb_fc[lgb_fc['origin'].isin(EVAL)]
static = ev[ev['variant'] == 'static'][['item_id', 'origin', 'date', 'forecast', 'actual']]
trunc = ev[ev['variant'] == 'adaptive'][['item_id', 'origin', 'date', 'forecast']].rename(columns={'forecast': 'truncate'})
wide = static.rename(columns={'forecast': 'static'}).merge(trunc, on=['item_id', 'origin', 'date'])
for v in ['decay', 'level_shift']:
    wide = wide.merge(new[new['variant'] == v][['item_id', 'origin', 'date', 'forecast']].rename(columns={'forecast': v}),
                      on=['item_id', 'origin', 'date'])
wide['blend'] = 0.5 * wide['static'] + 0.5 * wide['truncate']
wide = wide.merge(windows[['item_id', 'origin', 'has_break']], on=['item_id', 'origin'])
for v in ['decay', 'level_shift']:              # items without a break keep the static forecast
    wide.loc[~wide['has_break'], v] = wide.loc[~wide['has_break'], 'static']
wide.to_parquet("../data/processed/variant_forecasts.parquet", index=False)
print(len(wide), "rows")

2015-12-06 done (190s)


2016-01-03 done (394s)


2016-01-31 done (633s)


2016-02-28 done (892s)


2016-03-27 done (1302s)


2016-04-24 done (1450s)


121968 rows


## Scores and tests

In [5]:
VARIANTS = ['truncate', 'decay', 'level_shift', 'blend']
mae = wide.groupby(['item_id', 'origin']).apply(
    lambda g: pd.Series({v: np.mean(np.abs(g['actual'] - g[v])) for v in ['static'] + VARIANTS}), include_groups=False).reset_index()
mase = mae.merge(scales, on=['item_id', 'origin'])
for v in ['static'] + VARIANTS:
    mase[v] = mase[v] / mase['mase_scale']
chk = mase.merge(err[(err['model'] == 'lightgbm') & (err['variant'] == 'static')][['item_id', 'origin', 'mase']], on=['item_id', 'origin'])
print(f"static MASE vs 09a: max abs difference {np.abs(chk['static'] - chk['mase']).max():.2e}")

mase = mase.merge(windows[['item_id', 'origin', 'has_break']], on=['item_id', 'origin'])
mase = mase.merge(hc[['item_id', 'origin', 'high_confidence']], on=['item_id', 'origin'], how='left')
mase['high_confidence'] = mase['high_confidence'].fillna(False).astype(bool)
mase['category'] = mase['item_id'].map(category)
mase.to_csv("../outputs/tables/adaptation_variants_item_origin_mase.csv", index=False)

def item_diffs(mask, variant):
    s = mase[mask]
    return s.assign(diff=s[variant] - s['static']).groupby(['item_id', 'category'], as_index=False)['diff'].mean()

rows = []
for group, mask in [('all detected breaks', mase['has_break']), ('high-confidence', mase['high_confidence'])]:
    block = []
    for v in VARIANTS:
        d = item_diffs(mask, v)
        block.append({'group': group, 'variant': v, **paired_test(d['diff'])})
    adj = holm_adjust([b['p_value'] for b in block])
    for b, a in zip(block, adj):
        b['p_holm'] = a
        b['descriptive_only'] = b['n'] < 20
        b['helps'] = bool(not b['descriptive_only'] and a < 0.05 and b['median_diff'] < 0 and abs(b['median_diff']) >= PRACTICAL)
    rows += block
tests = pd.DataFrame(rows)
tests.to_csv("../outputs/tables/adaptation_variants_tests.csv", index=False)
pd.set_option('display.width', 220)
cols = ['group', 'variant', 'n', 'median_diff', 'ci_low', 'ci_high', 'share_improved', 'rank_biserial', 'p_value', 'p_holm', 'helps']
print(tests[cols].round(4).to_string(index=False))

static MASE vs 09a: max abs difference 1.33e-15


              group     variant   n  median_diff  ci_low  ci_high  share_improved  rank_biserial  p_value  p_holm  helps
all detected breaks    truncate 488       0.0033  0.0015   0.0053          0.4324         0.2340   0.0000  0.0000  False
all detected breaks       decay 488       0.0011  0.0004   0.0024          0.4406         0.1042   0.0460  0.1381  False
all detected breaks level_shift 488       0.0001 -0.0012   0.0006          0.4959        -0.0268   0.6084  0.6084  False
all detected breaks       blend 488      -0.0010 -0.0018  -0.0001          0.5533        -0.0835   0.1099  0.2197  False
    high-confidence    truncate 254       0.0033  0.0008   0.0074          0.4252         0.2070   0.0042  0.0169  False
    high-confidence       decay 254       0.0010 -0.0008   0.0026          0.4528         0.0480   0.5076  1.0000  False
    high-confidence level_shift 254       0.0007 -0.0008   0.0019          0.4685         0.0475   0.5115  1.0000  False
    high-confidence       blend 

In [6]:
# Descriptive: by category (not part of the pre-registered decision rule)
rows = []
for group, mask in [('all detected breaks', mase['has_break']), ('high-confidence', mase['high_confidence'])]:
    for v in VARIANTS:
        d = item_diffs(mask, v)
        for cat, g in d.groupby('category'):
            rows.append({'group': group, 'variant': v, 'category': cat, **paired_test(g['diff'])})
bycat = pd.DataFrame(rows)
bycat.to_csv("../outputs/tables/adaptation_variants_by_category.csv", index=False)
print(bycat[['group', 'variant', 'category', 'n', 'median_diff', 'ci_low', 'ci_high', 'p_value']].round(4).to_string(index=False))
print("\nH3 (no variant helps on all detected breaks):",
      "SUPPORTED" if not tests[tests['group'] == 'all detected breaks']['helps'].any() else "NOT SUPPORTED")

              group     variant  category   n  median_diff  ci_low  ci_high  p_value
all detected breaks    truncate     FOODS 234       0.0054  0.0017   0.0087   0.0001
all detected breaks    truncate   HOBBIES  67       0.0036  0.0008   0.0082   0.0059
all detected breaks    truncate HOUSEHOLD 187       0.0011 -0.0024   0.0046   0.1562
all detected breaks       decay     FOODS 234       0.0007 -0.0005   0.0035   0.3325
all detected breaks       decay   HOBBIES  67       0.0007 -0.0011   0.0033   0.4687
all detected breaks       decay HOUSEHOLD 187       0.0019  0.0005   0.0029   0.0611
all detected breaks level_shift     FOODS 234      -0.0014 -0.0027   0.0003   0.0954
all detected breaks level_shift   HOBBIES  67      -0.0011 -0.0028   0.0006   0.1960
all detected breaks level_shift HOUSEHOLD 187       0.0014  0.0002   0.0022   0.0400
all detected breaks       blend     FOODS 234      -0.0014 -0.0029   0.0005   0.4315
all detected breaks       blend   HOBBIES  67       0.0005 -0.000

## What we found

None of the four versions helps, on all shifts or on the high-confidence ones. First, the check worked: the
refactored LightGBM reproduces the 09a static forecasts exactly (largest difference 0.0).

On the 488 products with a shift, restarting from the shift is worse (+0.0033 MASE, Holm p < 0.001). Down-weighting
the old rows is slightly worse (+0.0011) and the shift feature does nothing (+0.0001). The blend is the only one that
points the right way, -0.0010 with a bootstrap interval just below zero (-0.0018 to -0.0001), but that is a seventh
of the 0.0074 we said a useful gain must be, and the Wilcoxon test is not significant after correcting for four
tests (p = 0.22). So by our own rule it does not help. On the 254 products with a high-confidence shift the
picture is the same.

So the answer is the same for machine learning as for the classical models: reacting to detected shifts does not
improve the forecast. LightGBM's 28- and 56-day averages already follow a level change on their own; our guess is
that anything that discards or down-weights history mostly loses the seasonal and price information in it.

## What comes next

`10_model_comparison` puts all models side by side on the same products.

In [7]:
# files this notebook wrote
from pathlib import Path
ROOT = Path('..').resolve()
written = [ROOT / p for p in ['outputs/tables/adaptation_variants_item_origin_mase.csv', 'outputs/tables/adaptation_variants_tests.csv', 'outputs/tables/adaptation_variants_by_category.csv', 'data/processed/variant_forecasts.parquet']]
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\adaptation_variants_item_origin_mase.csv 686 KB
outputs\tables\adaptation_variants_tests.csv 2 KB
outputs\tables\adaptation_variants_by_category.csv 5 KB
data\processed\variant_forecasts.parquet 6,272 KB
